# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [1]:
from typing_extensions import TypedDict

class NewsroomState(TypedDict):
    stories: dict   # keyed by story_id (slug), enriched in place by each agent

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [2]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [3]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

Stories fetched: 8
   262.6 vel | qwen-3-6-27b-is-the-sweet-spot-for-local-development
   120.7 vel | us-supreme-court-rules-geofence-warrants-require-constitutional-protections
   104.4 vel | rocketlab-acquires-iridium
    68.5 vel | a-native-graphical-shell-for-ssh
    35.8 vel | wataboy-jit-ing-game-boy-instructions-to-wasm-beats-a-native-interpreter
    22.6 vel | the-radiation-exposure-lie
    22.3 vel | ornith-1-0-self-improving-open-source-models-for-agentic-coding
     0.3 vel | wallace-the-6-inch-f-2-8-telescope-building-it-and-hiking-with-it


---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [4]:
import ollama, subprocess, time

# Health-check: Ollama must be running for Agent 2 synthesis
try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

✅ Ollama already running


In [5]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [13]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

Agent 2 Starts For : qwen-3-6-27b-is-the-sweet-spot-for-local-development
Trafiltura Success ✅  In Loading Content :7308 characters
  [background] gathering snippets for: Qwen 3.6 27B is the sweet spot for local development
[ddg] error for 'Qwen 3.6 27B is the sweet spot for local development': No results found.
  [bg] DDG returned 0 snippets
  [bg] Wikipedia search returns: empty (skipped)
  [background] 0 snippets, synthesizing (content-anchored)...
  [synth] 0 snippets, 104 snippet chars → groq/compound (will search)
  [synth] groq/compound-mini → 857 chars raw
Result After Background Syntezing is : Qwen (pronounced “queen”) is a family of large language models developed by Alibaba’s DAMO Academy and released under an open‑source Apache 2.0 license. The latest iteration, Qwen 3.6, arrived in early 2024 and offers two configurations: a dense 27‑billion‑parameter model and a 35‑billion‑parameter mixture‑of‑experts (MoE) version (often denoted A3B) that activates three expert pathways 

In [14]:
# Inspect Agent 2 output
for sid, story in call2["stories"].items():
    content_len  = len(story.get("content",  ""))
    bg_len       = len(story.get("background",""))
    bg_status    = f"{bg_len} chars" if bg_len else "EMPTY"
    print(f"{story['velocity']:>6.1f} vel | content {content_len:>5}c | bg {bg_status:<12} | {story['title'][:55]}")

 262.6 vel | content  6000c | bg 857 chars    | Qwen 3.6 27B is the sweet spot for local development
 120.7 vel | content  5961c | bg 697 chars    | US Supreme Court rules geofence warrants require consti
 104.4 vel | content  6000c | bg 716 chars    | Rocketlab acquires Iridium
  68.5 vel | content  3624c | bg 657 chars    | A native graphical shell for SSH
  35.8 vel | content  6000c | bg 1012 chars   | WATaBoy: JIT-Ing Game Boy Instructions to WASM Beats a 
  22.6 vel | content  6000c | bg 740 chars    | The Radiation Exposure Lie
  22.3 vel | content  6000c | bg 645 chars    | Ornith-1.0: self-improving open-source models for agent
   0.3 vel | content  6000c | bg 875 chars    | Wallace the 6 inch f/2.8 telescope, building it, and hi


---
## Agent 3: Fact Checker
Scores each story's credibility using two signals:

| Signal | Weight | How |
|--------|--------|-----|
|  | 25% | Domain trust map (github/arxiv/reuters etc.) |
|  | 75% | Groq gpt-oss-120b classifies: REAL / OPINION / SPAM |

**Why a 120B cloud model?**
The local 3B model judged credibility from *writing tone* alone — it wrongly labelled
real products (Haystack, Bunny DNS, Krea 2) as WEAK because they sounded promotional.
A 120B model has broad world knowledge to recognise these as genuine entities.

**Key design decisions:**
- REAL/OPINION/SPAM categories (not a decimal) — 3B+ models classify reliably but rate decimals poorly
- Soft score only — story is *marked* discarded, not deleted (audit trail; Agent 4 can override)
- Groq failure → 0.5 neutral (never discard due to an API crash)
- Thin content < 500 chars → 0.5 neutral (too little to judge reliably)

In [15]:
from agents.agent3 import source_score, llm_credibility_check, check_credibility
print("✅ Agent 3 imported")

✅ Agent 3 imported


In [16]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with credibility_score + discarded keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        print(f"{story['credibility_score']:.2f} {flag} | {story['title'][:55]}")
        time.sleep(1)
    return {"stories": stories}

In [20]:
# Run Agent 3 on Agent 2's output (no re-run of A1 or A2)
print("── CREDIBILITY RESULTS ─────────────────────────────")
call3 = fact_checker_node(call2)
print("────────────────────────────────────────────────────")

── CREDIBILITY RESULTS ─────────────────────────────
  [cred DEBUG] raw=''
  [cred] empty response → 0.5 neutral
0.50 ✅ KEEP | Qwen 3.6 27B is the sweet spot for local development
  [cred DEBUG] raw='REAL'
  [cred] US Supreme Court rules geofence warrants → 'REAL' → 0.9  (via openai/gpt-oss-120b)
0.80 ✅ KEEP | US Supreme Court rules geofence warrants require consti
  [cred DEBUG] raw='REAL'
  [cred] Rocketlab acquires Iridium → 'REAL' → 0.9  (via openai/gpt-oss-120b)
0.80 ✅ KEEP | Rocketlab acquires Iridium
  [cred DEBUG] raw='REAL'
  [cred] A native graphical shell for SSH → 'REAL' → 0.9  (via openai/gpt-oss-120b)
0.80 ✅ KEEP | A native graphical shell for SSH
  [cred DEBUG] raw='REAL'
  [cred] WATaBoy: JIT-Ing Game Boy Instructions t → 'REAL' → 0.9  (via openai/gpt-oss-120b)
0.80 ✅ KEEP | WATaBoy: JIT-Ing Game Boy Instructions to WASM Beats a 
  [cred DEBUG] raw=''
  [cred] empty response → 0.5 neutral
0.50 ✅ KEEP | The Radiation Exposure Lie
  [cred DEBUG] raw='REAL'
  [cred] Ornith

In [21]:
# Inspect all 3 agents output
print(f"\n{'FLAG':<4} {'SCORE':<6} {'VEL':>6} {'CONTENT':>9} {'BG':>6}  TITLE")
print("-" * 80)
for sid, story in call3["stories"].items():
    flag  = "🗑️" if story.get("discarded") else "✅"
    score = story.get("credibility_score", 0)
    vel   = story.get("velocity", 0)
    clen  = len(story.get("content", ""))
    bglen = len(story.get("background", ""))
    title = story.get("title", "")[:50]
    print(f"{flag}  {score:.2f}  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {title}")


FLAG SCORE     VEL   CONTENT     BG  TITLE
--------------------------------------------------------------------------------
✅  0.50   262.6     6000c   857c  Qwen 3.6 27B is the sweet spot for local developme
✅  0.80   120.7     5961c   697c  US Supreme Court rules geofence warrants require c
✅  0.80   104.4     6000c   716c  Rocketlab acquires Iridium
✅  0.80    68.5     3624c   657c  A native graphical shell for SSH
✅  0.80    35.8     6000c  1012c  WATaBoy: JIT-Ing Game Boy Instructions to WASM Bea
✅  0.50    22.6     6000c   740c  The Radiation Exposure Lie
✅  0.90    22.3     6000c   645c  Ornith-1.0: self-improving open-source models for 
✅  0.50     0.3     6000c   875c  Wallace the 6 inch f/2.8 telescope, building it, a


In [22]:
# ── INSPECT: full story details after all 3 agents ─────────────────────
for sid, story in call2["stories"].items():
    flag = "🗑️" if story.get("discarded") else "✅"
    print(f"{flag} {story['credibility_score']:.2f} | vel {story['velocity']:>6.1f} | "
          f"content {len(story.get('content','')):>5}c | "
          f"bg {len(story.get('background','')):>4}c | {story['title'][:50]}")

✅ 0.50 | vel  262.6 | content  6000c | bg  857c | Qwen 3.6 27B is the sweet spot for local developme
✅ 0.80 | vel  120.7 | content  5961c | bg  697c | US Supreme Court rules geofence warrants require c
✅ 0.80 | vel  104.4 | content  6000c | bg  716c | Rocketlab acquires Iridium
✅ 0.80 | vel   68.5 | content  3624c | bg  657c | A native graphical shell for SSH
✅ 0.80 | vel   35.8 | content  6000c | bg 1012c | WATaBoy: JIT-Ing Game Boy Instructions to WASM Bea
✅ 0.50 | vel   22.6 | content  6000c | bg  740c | The Radiation Exposure Lie
✅ 0.90 | vel   22.3 | content  6000c | bg  645c | Ornith-1.0: self-improving open-source models for 
✅ 0.50 | vel    0.3 | content  6000c | bg  875c | Wallace the 6 inch f/2.8 telescope, building it, a


In [24]:


# Save to disk — skip re-processing next time
from agents.story_cache import save_stories
save_stories(call3["stories"])

  [cache] saved 8 stories → data/stories_cache.json (8 total in cache)
